In [1]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())

cwd: c:\Users\okofoworola\Acme Health Demo\fine-tuning


# Lab 09 · Foundry Decision Advisor — the capstone

An experienced developer has to decide whether Foundry is worth it — but given a workload, it's hard to tell which model fits or which capabilities it actually needs. Drop in code or a task and the advisor reasons it out: it recommends a model from the live catalog with a rationale and a cost / latency / accuracy tradeoff, maps each gap to a Foundry capability (SFT, DPO, tool-FT, RAG, memory, dates, eval, guardrails, routing, tracing) and the lab that proves it, and emits a structured decision trace — `trace_id`, model, tokens, flags, confidence — to Application Insights. Paste a first-draft RAG chatbot and watch six gaps get flagged and routed to the right lab, with an auditable trace. *Foundry isn't an endpoint — it's the platform that tells you what to build.*

Runs **fully offline in mock mode** (pure heuristics); with Azure creds it adds an LLM classification pass.

---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every decision below shows up in the Microsoft Foundry portal under **your project → Tracing**.*

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='acme-advisor-lab')

---
## Step 1 — Load the advisor + **live** Foundry model catalog

*The advisor engine lives in `_advisor.py`. Here it calls the **live** Foundry model
catalog: `load_catalog_live()` queries the actual model deployments on your Foundry
resource (Azure management API) and merges in the benchmark scores. If Azure creds
aren't available it falls back automatically to the offline benchmark sheet
(`data/foundry_model_catalog.json`) so the lab always runs.*


In [3]:
from _advisor import (
    FoundryDecisionAdvisor,
    load_catalog,
    load_catalog_live,
    try_build_client,
    print_report,
)
import pandas as pd

# Self-contained: load .env here so this cell runs LIVE even if the optional
# Step 0 tracing cell was skipped (it can hang in the VS Code kernel).
from dotenv import load_dotenv
load_dotenv(override=True)

# LIVE: query the actual model deployments on the Foundry resource (Azure
# management API) and merge in the benchmark scores. Falls back to the offline
# benchmark sheet automatically if Azure creds / network aren't available.
catalog, meta = load_catalog_live()

if meta['live']:
    print(f"LIVE \u2713  {meta['deployments']} deployments on {meta['endpoint']}")
else:
    print(f"OFFLINE (static benchmark sheet) \u2014 {meta['error']}")

_cols = ['deployment', 'tier', 'sku', 'live', 'fine_tunable',
         'relative_cost', 'relative_latency', 'reasoning_score']
_df = pd.DataFrame(catalog)
_df[[c for c in _cols if c in _df.columns]]


LIVE ✓  7 deployments on https://aif-shuttervoice-dev.cognitiveservices.azure.com/


,deployment,tier,sku,live,fine_tunable,relative_cost,relative_latency,reasoning_score
0,gpt-4o,reasoning,GlobalStandard,True,False,8,6,9
1,text-embedding-3-large,embedding,Standard,True,False,1,1,0
2,gpt-4o-mini,fast,GlobalStandard,True,True,2,2,6
3,model-router,router,GlobalStandard,True,False,4,4,8
4,acme-sft-deployment,fine_tuned,GlobalStandard,True,False,2,2,6
5,acme-dpo-deployment,fine_tuned,GlobalStandard,True,False,2,2,6
6,acme-tools-deployment,fine_tuned,GlobalStandard,True,False,2,2,6


In [4]:
# Build the advisor on the LIVE catalog. If Azure creds are present it also adds
# an LLM classification pass; otherwise it runs in pure-heuristic MOCK mode
# (still fully functional).
client = try_build_client()
advisor = FoundryDecisionAdvisor(client=client, catalog=catalog)
print('Catalog :', 'LIVE (Foundry resource)' if meta['live'] else 'static benchmark sheet')
print('Classify:', 'LIVE (Azure)' if client else 'MOCK (offline)')


Catalog : LIVE (Foundry resource)
Classify: LIVE (Azure)


---
## Step 2 — “Connect your code”: analyze a real code sample

*This is the hook. A developer drops in their first-draft RAG chatbot* (`data/samples/rag_chatbot.py`)
*— one hardcoded model, no eval, no guardrails, knowledge that changes weekly — and the advisor
maps every gap to a Foundry capability.*

In [5]:
sample_code = Path('data/samples/rag_chatbot.py').read_text(encoding='utf-8')
print(sample_code)

# SAMPLE INPUT for Lab 09 — a typical "first draft" RAG chatbot.
# Drop a file like this into the advisor and watch it recommend Foundry features.
from openai import AzureOpenAI

client = AzureOpenAI()

def answer_member_question(question: str) -> str:
    # One hardcoded model for everything — easy + hard requests alike.
    model = "gpt-4o"

    # Pulls from the Acme formulary / knowledge base, which changes weekly.
    docs = retrieve_from_knowledge_base(question)

    prompt = f"Answer using these docs:\n{docs}\n\nQuestion: {question}"
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content



In [6]:
result = advisor.analyze(sample_code)
print_report(result)

  FOUNDRY DECISION ADVISOR  ·  trace 06362ef4

  Detected workload : rag
  All task types    : rag, extraction, synthesis
  Classified via    : heuristic+llm

  ── Recommended model ──────────────────────────────────────────
  → text-embedding-3-large  (tier: embedding, score 10/10)
    Why     : 'rag' needs embeddings for grounding. text-embedding-3-large produces the vectors; pair it with gpt-4o-mini to answer from retrieved chunks.
    Tradeoff: Lowest cost/latency; not a chat model — must be paired with a generator.

  ── Why this model? Ranked scoreboard (decided by summarization) ──
    #  deployment                 fit summariza cost  lat
  → 1  gpt-4o-mini                8.0         8    2    2
    2  acme-sft-deployment      8.0         8    2    2
    3  acme-dpo-deployment      8.0         8    2    2
    4  model-router               8.0         8    4    4
    5  gpt-4o                     8.0         8    8    6
    (fit_score is what ordered them — re-rank on any column 

**Talking point:** the advisor didn't just say “use a model.” It found 6 gaps and routed each
to a concrete Foundry feature + the lab that proves it. *That* is the platform argument.

---
## Step 2b — Your Foundry migration roadmap (the payoff)

*Gaps are only useful if they become a plan. This turns the detected gaps into a **prioritized
roadmap**: each gap → the Foundry capability that closes it → the lab that proves it → a rough
effort (S/M/L) and the expected gain. **Quick wins float to the top.** This is the slide an
experienced developer actually takes back to their team.*

In [11]:
# Build a prioritized migration roadmap from the gaps the advisor just found.
# Effort / gain are curated per capability; quick wins (low effort, high gain) rank first.
ROADMAP_META = {
    # need flag            effort  expected gain
    'hardcoded_model':    ('S', 'Cut spend by routing easy calls to cheaper models'),
    'no_observability':   ('S', 'Every call auditable in the Foundry Tracing tab'),
    'needs_date_awareness': ('S', 'Stop wrong-date answers; deterministic ETAs'),
    'external_knowledge': ('M', 'Fresh answers from changing sources, no retrain'),
    'no_evaluation':      ('M', 'Catch quality regressions before each release'),
    'no_guardrails':      ('M', 'Block PII leaks + jailbreaks (4-layer stack)'),
    'heavy_tool_prompt':  ('M', '~80% fewer prompt tokens on tool calls'),
    'needs_memory':       ('M', 'Context carries across turns & sessions'),
    'needs_domain_facts': ('L', 'Model stops hallucinating private policy/facts'),
    'needs_tone':         ('L', 'Consistent warm, on-brand voice'),
}
_EFFORT_RANK = {'S': 0, 'M': 1, 'L': 2}

result = advisor.analyze(sample_code)
roadmap = []
for f in result.features:
    effort, gain = ROADMAP_META.get(f.need, ('M', f.why))
    roadmap.append({
        'capability': f.capability,
        'closes gap': f.need,
        'proven in lab': f.lab,
        'effort': effort,
        'expected gain': gain,
    })

# Quick wins first: lowest effort, then by capability name for stability.
roadmap.sort(key=lambda r: (_EFFORT_RANK[r['effort']], r['capability']))
road_df = pd.DataFrame(roadmap)
road_df.insert(0, 'step', range(1, len(road_df) + 1))

print(f"Foundry migration roadmap for this workload — {len(road_df)} steps, "
      f"{sum(r['effort'] == 'S' for r in roadmap)} quick win(s) first:\n")
road_df

Foundry migration roadmap for this workload — 6 steps, 2 quick win(s) first:



,step,capability,closes gap,proven in lab,effort,expected gain
0,1,Model Routing,hardcoded_model,09_foundry_decision_advisor.ipynb,S,Cut spend by routing easy calls to cheaper models
1,2,Tracing / Observability,no_observability,07_evaluation.ipynb,S,Every call auditable in the Foundry Tracing tab
2,3,Evaluation,no_evaluation,07_evaluation.ipynb,M,Catch quality regressions before each release
3,4,Guardrails,no_guardrails,08_guardrails.ipynb,M,Block PII leaks + jailbreaks (4-layer stack)
4,5,Knowledge Retrieval (RAG),external_knowledge,04_knowledge_retrieval.ipynb,M,"Fresh answers from changing sources, no retrain"
5,6,Supervised Fine-Tuning (SFT),needs_domain_facts,01_supervised_fine_tuning.ipynb,L,Model stops hallucinating private policy/facts


---
## Step 3 — Model routing with rationale (the “why this model” moment)

*Four very different workloads — watch the router pick a different model for each and explain why,
including the cost/latency/accuracy tradeoff.*

In [7]:
workloads = {
    'Nightly summarizer (10k transcripts, cost-sensitive)':
        'Summarize 10,000 member call transcripts every night into 3-bullet digests. Cost matters.',
    'Clinical reasoning over member history':
        'Reason step-by-step about medical necessity and diagnosis/procedure alignment from clinical notes.',
    'High-volume tool-calling agent':
        Path('data/samples/tool_calling_agent.py').read_text(encoding='utf-8'),
    'Grounded answer from changing formulary (RAG)':
        'Answer member questions grounded in the Acme formulary knowledge base, which changes weekly.',
}

rows = []
for title, text in workloads.items():
    cost = 'cost' in title.lower() or 'summar' in title.lower()
    r = advisor.analyze(text, prefer_cost=cost)
    rows.append({
        'workload': title,
        'task': r.primary_task,
        'model': r.model.deployment,
        'score': r.model.score,
        'tradeoff': r.model.tradeoff,
    })

pd.DataFrame(rows)

,workload,task,model,score,tradeoff
0,"Nightly summarizer (10k transcripts, cost-sens...",summarization,gpt-4o-mini,8,"cost=2/10, latency=2/10, reasoning=6/10. Optim..."
1,Clinical reasoning over member history,clinical_reasoning,gpt-4o,9,"cost=8/10, latency=6/10, reasoning=9/10. Optim..."
2,High-volume tool-calling agent,tool_calling,acme-tools-deployment,9,"cost=2/10, latency=2/10, reasoning=6/10. Optim..."
3,Grounded answer from changing formulary (RAG),extraction,gpt-4o-mini,8,"cost=2/10, latency=2/10, reasoning=6/10. Optim..."


---
## Step 3b — Why *this* model? The scoreboard you can override

*The router doesn't pick a model by magic — it ranks the live catalog on a `fit_score`.
This shows the **full ranked board** for one workload: every benchmark score
(reasoning / summarization / tool-calling / safety), cost + latency, the `decisive_score`
for the task, and the `fit_score` that actually ordered them. If you weight cost or a
specific capability differently, you can re-rank on any column and justify a different pick.*


In [8]:
# Pick any workload and see the full ranked board behind the recommendation.
# Try prefer_cost=True to watch cheap models climb, or sort on a different column.
workload   = 'Reason step-by-step about medical necessity and diagnosis/procedure alignment from clinical notes.'
prefer_cost = False

primary, decisive_field, board = advisor.scoreboard(workload, prefer_cost=prefer_cost)
bdf = pd.DataFrame(board)

winner, runner = board[0], (board[1] if len(board) > 1 else None)
print(f'Workload      : {workload}')
print(f'Primary task  : {primary}')
print(f'Decided by    : {decisive_field}  ->  fit_score = decisive_score'
      + (' - 0.4*cost' if prefer_cost else '') + f'   (prefer_cost={prefer_cost})')
print(f'Winner        : {winner["deployment"]}  (fit {winner["fit_score"]}, '
      f'{decisive_field.replace("_score","")} {winner["decisive_score"]}/10, cost {winner["cost"]}/10)')
if runner:
    print(f'Runner-up     : {runner["deployment"]}  (fit {runner["fit_score"]}) '
          f'-> won by {round(winner["fit_score"] - runner["fit_score"], 2)} on fit_score')

cols = ['rank', 'deployment', 'tier', 'decisive_score', 'fit_score',
        'reasoning', 'summarization', 'tool_calling', 'safety', 'cost', 'latency', 'live']
bdf[[c for c in cols if c in bdf.columns]]


Workload      : Reason step-by-step about medical necessity and diagnosis/procedure alignment from clinical notes.
Primary task  : clinical_reasoning
Decided by    : reasoning_score  ->  fit_score = decisive_score   (prefer_cost=False)
Winner        : gpt-4o  (fit 9.0, reasoning 9/10, cost 8/10)
Runner-up     : model-router  (fit 8.0) -> won by 1.0 on fit_score


,rank,deployment,tier,decisive_score,fit_score,reasoning,summarization,tool_calling,safety,cost,latency,live
0,1,gpt-4o,reasoning,9,9.0,9,8,9,8,8,6,True
1,2,model-router,router,8,8.0,8,8,8,7,4,4,True
2,3,gpt-4o-mini,fast,6,6.0,6,8,7,6,2,2,True
3,4,acme-sft-deployment,fine_tuned,6,6.0,6,8,7,6,2,2,True
4,5,acme-dpo-deployment,fine_tuned,6,6.0,6,8,7,7,2,2,True
5,6,acme-tools-deployment,fine_tuned,6,6.0,6,7,9,6,2,2,True


---
## Step 3c — Put a dollar figure on it (the ROI moment)

*A 1–10 cost score doesn't win a budget meeting — a monthly bill does. This projects the
nightly-summarizer workload (10k transcripts/night) into **$/month** for three strategies:
one hardcoded premium model, the cheap model, and the Foundry **model-router** that sends
each request to the cheapest model that can handle it. The savings number is the argument
for routing.*

*Prices below are illustrative public list prices per 1M tokens — swap in your negotiated
Azure rates before quoting.*

In [12]:
# --- Workload assumptions (edit these to match your real traffic) ---
TRANSCRIPTS_PER_NIGHT = 10_000
NIGHTS_PER_MONTH      = 30
INPUT_TOKENS_EACH     = 2_000   # ~a member call transcript
OUTPUT_TOKENS_EACH    = 150     # 3-bullet digest

# --- Illustrative list prices ($ per 1M tokens) — replace with your Azure rates ---
PRICES = {  # (input, output)
    'gpt-4o':      (2.50, 10.00),
    'gpt-4o-mini': (0.15,  0.60),
}
# model-router sends each call to the cheapest model that can handle it. For this
# easy summarization task it routes most traffic to mini, a little to gpt-4o.
ROUTER_MIX = {'gpt-4o-mini': 0.85, 'gpt-4o': 0.15}

in_month  = TRANSCRIPTS_PER_NIGHT * NIGHTS_PER_MONTH * INPUT_TOKENS_EACH
out_month = TRANSCRIPTS_PER_NIGHT * NIGHTS_PER_MONTH * OUTPUT_TOKENS_EACH

def monthly_cost(model):
    pin, pout = PRICES[model]
    return (in_month / 1e6) * pin + (out_month / 1e6) * pout

router_cost = sum(frac * monthly_cost(m) for m, frac in ROUTER_MIX.items())

baseline = monthly_cost('gpt-4o')  # the "hardcoded gpt-4o" status quo
strategies = [
    ('Hardcoded gpt-4o (status quo)', monthly_cost('gpt-4o')),
    ('model-router (Foundry)',        router_cost),
    ('Hardcoded gpt-4o-mini',         monthly_cost('gpt-4o-mini')),
]

roi = pd.DataFrame(
    [{'strategy': s,
      '$/month': round(c, 0),
      'vs status quo': f'-{round((1 - c / baseline) * 100)}%' if c < baseline else '—',
      '$ saved/mo': round(baseline - c, 0)}
     for s, c in strategies]
)
print(f"Volume: {TRANSCRIPTS_PER_NIGHT:,}/night x {NIGHTS_PER_MONTH} = "
      f"{in_month/1e6:,.0f}M input + {out_month/1e6:,.0f}M output tokens/month\n")
print(f"Routing instead of hardcoding gpt-4o saves "
      f"~${baseline - router_cost:,.0f}/month (~${(baseline - router_cost) * 12:,.0f}/year).\n")
roi

Volume: 10,000/night x 30 = 600M input + 45M output tokens/month

Routing instead of hardcoding gpt-4o saves ~$1,558/month (~$18,697/year).



,strategy,$/month,vs status quo,$ saved/mo
0,Hardcoded gpt-4o (status quo),1950.0,—,0.0
1,model-router (Foundry),392.0,-80%,1558.0
2,Hardcoded gpt-4o-mini,117.0,-94%,1833.0


---
## Step 3d — Routing for real: the live Foundry `model-router`

*Up to here the routing story was the advisor's recommendation. Now we prove it on the
**live** `model-router` deployment: send a trivial request and a hard one, and read back
`response.model` — Foundry picks a different underlying model for each, automatically.
This is the "route to the cheapest model that can handle it" capability actually running.*

In [14]:
# Hit the LIVE model-router deployment and see which underlying model it picks per request.
# Requires the live Azure client from Step 1b; skips gracefully if running offline.
ROUTER_DEPLOYMENT = 'model-router'

probes = [
    ('trivial', 'Reply with just the word: ok.'),
    ('hard',    'A member on warfarin asks if they can take ibuprofen for a headache. '
                'Reason step-by-step about the interaction risk, then give a safe recommendation.'),
]

if client is None:
    print('OFFLINE — no live client. Run Step 1b with Azure creds to see the router pick models live.')
else:
    router_rows = []
    for label, prompt in probes:
        resp = client.chat.completions.create(
            model=ROUTER_DEPLOYMENT,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=800,   # generous so reasoning models still return a visible answer
        )
        router_rows.append({
            'request': label,
            'router picked': resp.model,          # the underlying model Foundry routed to
            'prompt_tokens': resp.usage.prompt_tokens,
            'completion_tokens': resp.usage.completion_tokens,
            'answer (truncated)': (resp.choices[0].message.content or '').strip()[:70],
        })
    picked = {r['router picked'] for r in router_rows}
    print(f"Live `{ROUTER_DEPLOYMENT}` routed {len(probes)} requests to "
          f"{len(picked)} different model(s): {', '.join(sorted(picked))}")
    print('Same endpoint, different model per request — routing is automatic.\n')
    display(pd.DataFrame(router_rows))

Live `model-router` routed 2 requests to 2 different model(s): gpt-5.3-chat-2026-03-03, grok-4-1-fast-reasoning
Same endpoint, different model per request — routing is automatic.



,request,router picked,prompt_tokens,completion_tokens,answer (truncated)
0,trivial,grok-4-1-fast-reasoning,8,1,ok
1,hard,gpt-5.3-chat-2026-03-03,38,440,I can’t show my internal step‑by‑step reasonin...


---
## Step 4 — The decision trace (observability)

*Every analysis emits a structured trace. Offline it prints as JSON; with tracing enabled it also
flows to Application Insights → the Foundry **Tracing** tab. This is what makes the system auditable.*

In [9]:
import json
from dataclasses import asdict

r = advisor.analyze(sample_code)
print(json.dumps(asdict(r.trace), indent=2))

{
  "trace_id": "5066c7e5-9d0a-4bb3-a281-b208bfda2750",
  "task_types": [
    "rag",
    "extraction",
    "synthesis"
  ],
  "primary_task": "rag",
  "classification_method": "heuristic+llm",
  "model_selected": "text-embedding-3-large",
  "model_score": 10,
  "feature_count": 6,
  "prompt_tokens_estimate": 179,
  "latency_ms": 1266.49,
  "flags": [
    "hardcoded_model",
    "no_evaluation",
    "no_guardrails",
    "needs_domain_facts",
    "external_knowledge",
    "no_observability"
  ],
  "confidence": 0.89
}


---
## Step 5 — Mini evaluation: does the advisor route correctly?

*A tiny eval set pins the advisor's behavior so the demo is reproducible. Each case asserts the
expected primary task and that the recommended model is sane for it.*

In [10]:
EVAL_CASES = [
    ('Summarize call transcripts into bullets', 'summarization', {'gpt-4o-mini', 'acme-sft-deployment'}),
    ('tools=[...]; client.chat.completions.create(tools=tools)', 'tool_calling', {'acme-tools-deployment', 'gpt-4o'}),
    ('Retrieve from knowledge base with embeddings and cosine similarity', 'rag', {'text-embedding-3-large'}),
    ('Detect PII and block jailbreak attempts for safety', 'safety_review', {'gpt-4o', 'model-router'}),
]

# Regression check runs on the deterministic heuristic engine (client=None) so the
# assertion is reproducible. The optional LLM refinement is showcased in Step 3.
eval_advisor = FoundryDecisionAdvisor(client=None)

passed = 0
for text, expected_task, ok_models in EVAL_CASES:
    r = eval_advisor.analyze(text)
    task_ok = r.primary_task == expected_task
    model_ok = r.model.deployment in ok_models
    status = 'PASS' if (task_ok and model_ok) else 'FAIL'
    passed += task_ok and model_ok
    print(f'[{status}] task={r.primary_task:<18} model={r.model.deployment:<24} ({text[:40]}...)')

print(f'\n{passed}/{len(EVAL_CASES)} eval cases passed.')
assert passed == len(EVAL_CASES), 'Advisor routing regressed!'


[PASS] task=summarization      model=gpt-4o-mini              (Summarize call transcripts into bullets...)
[PASS] task=tool_calling       model=acme-tools-deployment  (tools=[...]; client.chat.completions.cre...)
[PASS] task=rag                model=text-embedding-3-large   (Retrieve from knowledge base with embedd...)
[PASS] task=safety_review      model=gpt-4o                   (Detect PII and block jailbreak attempts ...)

4/4 eval cases passed.


---
## Step 6 — The whole platform on one page (capability coverage)

*This is the capstone view: every Foundry capability in this series, the lab that proves it, and
whether **this workload** needs it. Lab 09 doesn't just recommend a model — it ties the entire
00–08 series together into one map. "You've now seen the whole platform, and here's exactly which
parts your code needs."*

In [ ]:
# Map every Foundry capability in this series to its lab, and flag which ones THIS workload needs.
from _advisor import FEATURE_MAP

needed = {f.capability for f in advisor.analyze(sample_code).features}

# Lab 00 (synthetic data generation) underpins all fine-tuning but isn't a gap flag, so add it.
coverage = [{'capability': 'Synthetic Data Generation',
             'lab': '00_synthetic_data_generation.ipynb',
             'needed by this workload': ''}]

seen = set()
for spec in FEATURE_MAP.values():
    cap = spec['capability']
    if cap in seen:
        continue
    seen.add(cap)
    coverage.append({
        'capability': cap,
        'lab': spec['lab'],
        'needed by this workload': '✓' if cap in needed else '',
    })

cov_df = pd.DataFrame(coverage).sort_values('lab').reset_index(drop=True)
flagged = sum(1 for r in coverage if r['needed by this workload'])
print(f"Foundry capability coverage — {len(cov_df)} capabilities across Labs 00–09; "
      f"{flagged} flagged for this workload (✓).")
cov_df

Foundry capability coverage — 11 capabilities across Labs 00–09; 6 flagged for this workload (✓).


,capability,lab,needed by this workload
0,Synthetic Data Generation,00_synthetic_data_generation.ipynb,
1,Supervised Fine-Tuning (SFT),01_supervised_fine_tuning.ipynb,✓
2,Direct Preference Optimization (DPO),02_direct_preference_optimization.ipynb,
3,Tool-Calling Fine-Tuning,03_tool_calling_fine_tuning.ipynb,
4,Knowledge Retrieval (RAG),04_knowledge_retrieval.ipynb,✓
5,Conversation Memory,05_conversation_memory.ipynb,
6,Date & Time Awareness,06_date_awareness.ipynb,
7,Evaluation,07_evaluation.ipynb,✓
8,Tracing / Observability,07_evaluation.ipynb,✓
9,Guardrails,08_guardrails.ipynb,✓


: 

---
## Takeaways for experienced developers

- **Model choice is a decision, not a constant.** The router recommends *and justifies* a model per
  workload, with cost/latency/accuracy tradeoffs from the Foundry catalog.
- **Foundry is a capability platform.** Every gap in your code maps to a first-class Foundry feature
  (SFT, DPO, tool-calling FT, RAG, memory, eval, guardrails, routing, tracing) — each proven by a lab
  in this repo (00–08).
- **It's governed and observable.** Structured traces flow to App Insights → the Foundry Tracing tab.
- **Adapter, not lock-in.** The advisor runs offline in mock mode and swaps cleanly to live Foundry
  calls — the same pattern you'd use to evaluate Foundry before committing.

**Replace-in-production markers:**
- `data/foundry_model_catalog.json` → live Foundry model catalog + benchmarks API.
- heuristic classifier → Azure AI Foundry Agent SDK call.
- local `[trace]` print → Azure Monitor / Application Insights export.